# Feature Engineering, Missing Data, Scaling, Cross-validation, Hyperparameter Tuning

Notebook này dùng để ghi chú và thực hành các phần chính của tuần 16:

- Feature engineering
- Xử lý missing data
- Normalization, scaling
- Cross-validation
- Hyperparameter tuning

Mục tiêu chính: hiểu cách chuẩn bị dữ liệu trước khi đưa vào model và cách đánh giá/tối ưu model đúng cách.

## 0. Pipeline tổng quát trong Machine Learning

Một pipeline thực tế thường có dạng:

```text
Raw data
→ Feature engineering
→ Xử lý missing data
→ Encoding / Scaling / Normalization
→ Train model
→ Cross-validation
→ Hyperparameter tuning
→ Final evaluation
```

Nói đơn giản:

- **Feature engineering**: làm dữ liệu thông minh hơn.
- **Missing data**: xử lý dữ liệu bị thiếu.
- **Scaling / Normalization**: đưa dữ liệu về cùng thang đo.
- **Cross-validation**: kiểm tra model nhiều lần cho chắc.
- **Hyperparameter tuning**: chỉnh các tham số điều khiển để model tốt hơn.

# 1. Feature Engineering

## 1.1 Feature Engineering là gì?

**Feature engineering** là quá trình tạo mới, biến đổi hoặc chọn lọc các đặc trưng để giúp model học tốt hơn.

Trong Machine Learning:

```text
Feature = cột dữ liệu dùng để dự đoán
Target = cột kết quả cần dự đoán
```

Ví dụ bài toán dự đoán khách hàng có rời bỏ dịch vụ hay không:

| Age | MonthlyCharge | ContractType | Churn |
|---:|---:|---|---|
| 25 | 20 | Monthly | Yes |
| 40 | 80 | Yearly | No |

Trong bảng trên:

```text
Age, MonthlyCharge, ContractType = features
Churn = target
```

## 1.2 Vì sao cần Feature Engineering?

Dữ liệu gốc đôi khi chưa đủ rõ ý nghĩa cho model.

Ví dụ có cột:

```text
DateOfBirth = 2001-05-20
```

Model sẽ khó hiểu trực tiếp ý nghĩa của ngày sinh. Nhưng nếu biến đổi thành:

```text
Age = 25
```

thì model dễ học hơn.

Đây chính là feature engineering.

## 1.3 Một số kỹ thuật Feature Engineering phổ biến

### 1. Tạo feature mới

Ví dụ dataset bán hàng có:

```text
Quantity
Price
```

Ta có thể tạo:

```text
TotalAmount = Quantity × Price
```

Feature mới này thường có ý nghĩa hơn từng cột riêng lẻ.

---

### 2. Tách feature từ ngày tháng

Từ cột:

```text
OrderDate = 2026-06-14
```

Có thể tách thành:

```text
Year = 2026
Month = 6
Day = 14
DayOfWeek = Sunday
IsWeekend = True
```

Cách này thường dùng trong ecommerce, bán lẻ, dự báo doanh thu, phân tích hành vi khách hàng.

---

### 3. Biến đổi categorical data

Ví dụ:

```text
Gender = Male / Female
City = HCM / Hanoi / Da Nang
```

Model thường không hiểu chữ trực tiếp nên cần encoding.

Hai cách phổ biến:

```text
Label Encoding
One-Hot Encoding
```

Ví dụ One-Hot Encoding:

| City | City_HCM | City_Hanoi | City_DaNang |
|---|---:|---:|---:|
| HCM | 1 | 0 | 0 |
| Hanoi | 0 | 1 | 0 |
| Da Nang | 0 | 0 | 1 |

---

### 4. Binning

Binning là chia giá trị liên tục thành nhóm.

Ví dụ tuổi:

```text
0 - 18     → Teen
19 - 35    → Young Adult
36 - 60    → Adult
60+        → Senior
```

---

### 5. Feature interaction

Tạo feature từ sự kết hợp giữa các feature khác.

Ví dụ:

```text
Income
Debt
```

Có thể tạo:

```text
DebtRatio = Debt / Income
```

Trong thực tế, `DebtRatio` có thể có ý nghĩa hơn `Debt` hoặc `Income` đứng riêng.

# 2. Xử lý Missing Data

## 2.1 Missing Data là gì?

**Missing data** là dữ liệu bị thiếu.

Ví dụ:

| Age | Salary | City |
|---:|---:|---|
| 25 | 1000 | HCM |
| NaN | 1500 | Hanoi |
| 30 | NaN | Da Nang |

`NaN` nghĩa là giá trị bị thiếu.

## 2.2 Vì sao missing data nguy hiểm?

Nhiều thuật toán Machine Learning không xử lý được `NaN`.

Ví dụ:

```python
LinearRegression()
LogisticRegression()
SVM()
KNN()
```

Các model này thường yêu cầu dữ liệu đầu vào phải đầy đủ.

## 2.3 Các cách xử lý missing data

### Cách 1: Xóa dòng bị thiếu

```python
df.dropna()
```

Dùng khi số dòng bị thiếu rất ít.

Ví dụ dataset có 10,000 dòng, chỉ 20 dòng bị thiếu thì có thể xóa.

Nhược điểm: nếu thiếu nhiều mà xóa hết thì mất dữ liệu.

---

### Cách 2: Điền bằng mean

Dùng cho dữ liệu số.

```python
df["Age"].fillna(df["Age"].mean())
```

Ví dụ:

```text
Age = [20, 30, NaN, 40]
Mean = (20 + 30 + 40) / 3 = 30
```

Sau khi điền:

```text
Age = [20, 30, 30, 40]
```

---

### Cách 3: Điền bằng median

Median là giá trị ở giữa.

Dùng khi dữ liệu có outlier.

Ví dụ:

```text
Salary = [500, 600, 700, 100000]
```

Mean sẽ bị kéo lên rất cao vì `100000`.

Trong trường hợp này, dùng median tốt hơn.

---

### Cách 4: Điền bằng mode

Mode là giá trị xuất hiện nhiều nhất.

Dùng cho categorical data.

Ví dụ:

```text
City = [HCM, HCM, Hanoi, NaN]
```

Mode là `HCM`.

---

### Cách 5: Thêm cột đánh dấu missing

Đôi khi việc một giá trị bị thiếu cũng mang ý nghĩa.

Ví dụ khách hàng không nhập số điện thoại có thể là dấu hiệu họ ít tin tưởng hệ thống.

Ta có thể tạo:

```text
Phone_missing = 1 nếu thiếu
Phone_missing = 0 nếu không thiếu
```

# 3. Normalization và Scaling

## 3.1 Scaling là gì?

**Scaling** là đưa các feature số về cùng một thang đo.

Ví dụ:

| Age | Salary |
|---:|---:|
| 25 | 1000 |
| 30 | 2000 |
| 35 | 5000 |

`Salary` lớn hơn `Age` rất nhiều. Một số thuật toán có thể hiểu nhầm rằng `Salary` quan trọng hơn chỉ vì giá trị lớn hơn.

## 3.2 Khi nào cần scaling?

Các thuật toán rất cần scaling:

```text
KNN
SVM
Logistic Regression
Linear Regression dùng Gradient Descent
K-Means
PCA
Neural Network
```

Các thuật toán ít cần scaling hơn:

```text
Decision Tree
Random Forest
XGBoost / LightGBM / CatBoost
```

Lý do: tree-based model chia nhánh theo điều kiện như:

```text
Age <= 30
Salary <= 5000
```

nên không quá nhạy với scale.

## 3.3 Standardization

Công thức:

$$
z = \frac{x - \mu}{\sigma}
$$

Trong đó:

```text
x  = giá trị ban đầu
μ  = mean
σ  = standard deviation
z  = giá trị sau khi chuẩn hóa
```

Sau standardization, dữ liệu thường có:

```text
Mean ≈ 0
Standard deviation ≈ 1
```

Ví dụ dùng trong Python:

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```

## 3.4 Min-Max Normalization

Công thức:

$$
x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

Kết quả thường nằm trong khoảng:

```text
0 đến 1
```

Ví dụ:

```text
Age = [20, 30, 40]
```

Với:

```text
min = 20
max = 40
```

Ta có:

$$
Age_{20} = \frac{20 - 20}{40 - 20} = 0
$$

$$
Age_{30} = \frac{30 - 20}{40 - 20} = 0.5
$$

$$
Age_{40} = \frac{40 - 20}{40 - 20} = 1
$$

Ví dụ dùng trong Python:

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
```

## 3.5 StandardScaler vs MinMaxScaler vs RobustScaler

| Kỹ thuật | Kết quả | Khi nào dùng |
|---|---|---|
| StandardScaler | Mean 0, std 1 | Logistic Regression, SVM, PCA, K-Means |
| MinMaxScaler | Giá trị từ 0 đến 1 | Neural Network, dữ liệu cần giới hạn range |
| RobustScaler | Dùng median, IQR | Khi dữ liệu có outlier |

# 4. Cross-validation

## 4.1 Cross-validation là gì?

**Cross-validation** là kỹ thuật chia dữ liệu thành nhiều phần để đánh giá model ổn định hơn.

Thay vì chỉ chia một lần:

```text
Train 80% - Test 20%
```

Ta chia nhiều lần.

Ví dụ **5-fold cross-validation**:

```text
Fold 1: Test phần 1, train phần còn lại
Fold 2: Test phần 2, train phần còn lại
Fold 3: Test phần 3, train phần còn lại
Fold 4: Test phần 4, train phần còn lại
Fold 5: Test phần 5, train phần còn lại
```

Sau đó lấy trung bình điểm số.

## 4.2 Vì sao cần cross-validation?

Vì một lần train/test split có thể bị may rủi.

Ví dụ:

```text
Lần 1 accuracy = 90%
Lần 2 accuracy = 75%
Lần 3 accuracy = 82%
```

Nếu chỉ nhìn một lần thì dễ đánh giá sai model.

Cross-validation giúp ta biết model có ổn định không.

## 4.3 Các loại cross-validation phổ biến

| Loại | Ý nghĩa |
|---|---|
| K-Fold | Chia dữ liệu thành K phần |
| Stratified K-Fold | Giữ tỷ lệ class giống nhau ở mỗi fold |
| Leave-One-Out | Mỗi lần test 1 sample |
| TimeSeriesSplit | Dùng cho dữ liệu theo thời gian |

Với bài toán classification, thường dùng:

```python
StratifiedKFold
```

vì nó giữ tỷ lệ class cân bằng giữa các fold.

# 5. Hyperparameter Tuning

## 5.1 Hyperparameter là gì?

Trong Machine Learning có 2 khái niệm dễ nhầm:

| Khái niệm | Ý nghĩa |
|---|---|
| Parameter | Model tự học từ dữ liệu |
| Hyperparameter | Mình phải chọn trước khi train |

Ví dụ với Logistic Regression:

```text
weights, bias = parameter
C, penalty, solver = hyperparameter
```

Ví dụ với Random Forest:

```text
n_estimators
max_depth
min_samples_split
```

Đây là hyperparameter.

## 5.2 Hyperparameter tuning là gì?

**Hyperparameter tuning** là quá trình thử nhiều bộ hyperparameter khác nhau để tìm ra bộ tốt nhất.

Ví dụ Random Forest:

```text
n_estimators = 100, 200, 300
max_depth = 5, 10, 20
```

Model sẽ thử nhiều tổ hợp rồi chọn tổ hợp có kết quả tốt nhất.

## 5.3 Grid Search

Grid Search thử tất cả các tổ hợp.

Ví dụ:

```python
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None]
}
```

Số tổ hợp:

```text
2 × 3 = 6 tổ hợp
```

Ưu điểm:

```text
Tìm kỹ
```

Nhược điểm:

```text
Chạy chậm nếu nhiều tham số
```

## 5.4 Random Search

Random Search chọn ngẫu nhiên một số tổ hợp để thử.

Ưu điểm:

```text
Nhanh hơn Grid Search
```

Nhược điểm:

```text
Có thể bỏ sót tổ hợp tốt nhất
```

Dùng khi không gian tìm kiếm quá lớn.

# 6. Demo tổng hợp bằng Python

Trong demo này ta sẽ dùng dataset `Breast Cancer` có sẵn trong `sklearn`.

Mục tiêu của demo:

1. Load dữ liệu thật có sẵn trong thư viện.
2. Tạo thêm feature mới.
3. Giả lập missing data để thực hành xử lý.
4. Dùng `Pipeline` để tránh data leakage.
5. Dùng `Cross-validation`.
6. Dùng `GridSearchCV` để tuning hyperparameter.

In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
# Load dataset
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Shape X:", X.shape)
print("Shape y:", y.shape)
X.head()

Shape X: (569, 30)
Shape y: (569,)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 6.1 Feature Engineering trong demo

Ta sẽ tạo một feature mới:

```text
mean radius / mean area
```

Ý tưởng: thay vì chỉ dùng `mean radius` và `mean area` riêng lẻ, ta tạo thêm một tỷ lệ để model có thêm thông tin.

In [4]:
# Tạo feature mới
X["radius_area_ratio"] = X["mean radius"] / (X["mean area"] + 1e-9)

X[["mean radius", "mean area", "radius_area_ratio"]].head()

,mean radius,mean area,radius_area_ratio
0,17.99,1001.0,0.017972
1,20.57,1326.0,0.015513
2,19.69,1203.0,0.016367
3,11.42,386.1,0.029578
4,20.29,1297.0,0.015644


## 6.2 Giả lập missing data

Dataset gốc khá sạch, nên ta sẽ cố tình tạo một ít missing values để thực hành.

In [5]:
# Copy dữ liệu để tránh làm thay đổi dữ liệu gốc
X_missing = X.copy()

rng = np.random.default_rng(42)

# Chọn ngẫu nhiên 5% ô dữ liệu để biến thành NaN
mask = rng.random(X_missing.shape) < 0.05
X_missing = X_missing.mask(mask)

print("Tổng số missing values:", X_missing.isna().sum().sum())
X_missing.head()

Tổng số missing values: 873


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,radius_area_ratio
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,NaN,0.4601,0.11890,0.017972
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0.015513
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,NaN,0.12790,0.2069,0.05999,...,25.53,NaN,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0.016367
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0.029578
4,NaN,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,NaN,0.4000,0.1625,0.2364,0.07678,0.015644


## 6.3 Chia train/test

Lưu ý:

- `X_train` dùng để train model.
- `X_test` dùng để đánh giá cuối cùng.
- Không được để thông tin từ test set rò rỉ vào quá trình train.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_missing,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (455, 31)
Test shape: (114, 31)


## 6.4 Tạo Pipeline

Pipeline gồm 3 bước:

```text
SimpleImputer(strategy="median")
→ StandardScaler()
→ LogisticRegression()
```

Ý nghĩa:

- `SimpleImputer`: xử lý missing data bằng median.
- `StandardScaler`: chuẩn hóa dữ liệu.
- `LogisticRegression`: model phân loại.

In [7]:
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model

,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


## 6.5 Cross-validation

Ta dùng 5-fold cross-validation để kiểm tra model ổn định không.

In [8]:
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("CV scores:", scores)
print("Mean CV accuracy:", scores.mean())
print("Std CV accuracy:", scores.std())

CV scores: [0.95604396 0.97802198 0.96703297 1.         0.98901099]
Mean CV accuracy: 0.9780219780219781
Std CV accuracy: 0.015540808377726312


## 6.6 Hyperparameter Tuning bằng GridSearchCV

Với Logistic Regression, ta tuning tham số `C`.

Ý nghĩa đơn giản:

```text
C nhỏ  → regularization mạnh hơn
C lớn  → regularization yếu hơn
```

Regularization giúp hạn chế overfitting.

In [9]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__penalty": ["l2"],
    "classifier__solver": ["lbfgs"]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'classifier__C': 1, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
Best CV score: 0.9780219780219781


## 6.7 Đánh giá trên test set

Sau khi chọn được model tốt nhất từ cross-validation, ta đánh giá cuối cùng trên test set.

In [10]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

Test accuracy: 0.9736842105263158

Confusion matrix:
[[41  1]
 [ 2 70]]

Classification report:
              precision    recall  f1-score   support

   malignant       0.95      0.98      0.96        42
      benign       0.99      0.97      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



# 7. Data Leakage

## 7.1 Data leakage là gì?

**Data leakage** là khi thông tin từ test set bị rò rỉ vào quá trình train.

Đây là lỗi rất nguy hiểm vì làm kết quả đánh giá nhìn có vẻ tốt nhưng thực tế model không tốt như vậy.

## 7.2 Ví dụ sai

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

Sai vì scaler đã học mean/std từ toàn bộ dataset, bao gồm cả test set.

## 7.3 Cách đúng

```python
X_train, X_test, y_train, y_test = train_test_split(X, y)

scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

Cách tốt hơn nữa là dùng:

```python
Pipeline
```

Vì pipeline giúp các bước xử lý được thực hiện đúng thứ tự, tránh rò rỉ dữ liệu.

# 8. Tổng kết tuần 16

| Phần | Ý nghĩa chính | Ví dụ |
|---|---|---|
| Feature engineering | Tạo/biến đổi feature để model học tốt hơn | `TotalAmount = Price × Quantity` |
| Missing data | Xử lý dữ liệu bị thiếu | Điền mean, median, mode |
| Normalization/Scaling | Đưa dữ liệu về cùng thang đo | StandardScaler, MinMaxScaler |
| Cross-validation | Đánh giá model nhiều lần cho ổn định | 5-fold CV |
| Hyperparameter tuning | Tìm cấu hình model tốt nhất | GridSearchCV, RandomizedSearchCV |

## Cách nhớ nhanh

```text
Feature Engineering = làm dữ liệu thông minh hơn

Missing Data = vá lỗ hổng dữ liệu

Scaling = đưa dữ liệu về cùng thước đo

Cross-validation = kiểm tra model nhiều lần cho chắc

Hyperparameter Tuning = chỉnh nút vặn để model chạy tốt hơn
```

Nói kiểu thực tế:

```text
Feature engineering là chuẩn bị nguyên liệu.
Missing data là nhặt sạn.
Scaling là cân đo lại nguyên liệu.
Cross-validation là nếm thử nhiều lần.
Hyperparameter tuning là chỉnh công thức nấu.
```

Model ngon hay không, nhiều khi nằm ở mấy bước này chứ không chỉ ở thuật toán.

# 9. Bài tập tự luyện

## Bài 1

Cho dataset bán hàng có các cột:

```text
Price
Quantity
Discount
CustomerAge
OrderDate
```

Hãy tạo thêm ít nhất 3 feature mới.

Gợi ý:

```text
TotalAmount = Price × Quantity
FinalAmount = Price × Quantity × (1 - Discount)
OrderMonth
IsWeekend
AgeGroup
```

---

## Bài 2

Cho cột `Age` có missing values.

Hãy thử 3 cách xử lý:

```text
dropna()
fill bằng mean
fill bằng median
```

Sau đó so sánh số lượng dòng còn lại và giá trị trung bình của cột Age.

---

## Bài 3

Tạo một dataset nhỏ có 2 cột:

```text
Age
Salary
```

Sau đó thử:

```text
StandardScaler
MinMaxScaler
RobustScaler
```

Quan sát kết quả khác nhau như thế nào.

---

## Bài 4

Dùng một model bất kỳ, ví dụ Logistic Regression hoặc KNN.

So sánh kết quả:

```text
Train/test split một lần
5-fold cross-validation
```

---

## Bài 5

Dùng `GridSearchCV` để tuning một trong các model sau:

```text
Logistic Regression
KNN
Random Forest
SVM
```

Ghi lại:

```text
Best parameters
Best CV score
Test accuracy
```